In [82]:
import pandas as pd
import numpy as np
import ast

movies = pd.read_csv('../data/tmdb_5000_movies.csv')
credits = pd.read_csv('../data/tmdb_5000_credits.csv')

movies = pd.merge(movies,credits, on='title')

movies_df = movies[['id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']].copy()

movies_df.dropna(inplace=True)

def convert(text):
    l = ast.literal_eval(text)
    return [i['name'] for i in l]

movies_df['genres'] = movies_df['genres'].apply(convert)

movies_df['keywords'] = movies_df['keywords'].apply(convert)

movies_df['cast'] = movies_df['cast'].apply(convert)
movies_df['cast'] = movies_df['cast'].apply(lambda x: x[:3])


def fetch_director(text):
    l = ast.literal_eval(text)
    for i in l:
        if i['job'] == 'Director':
            return [i['name']]

movies_df['crew'] = movies_df['crew'].apply(fetch_director)

def remove_space(text):
    l = text
    for i in range(len(l)):
        result = ''
        s = l[i].split()
        for j in s:
            result += j
        l[i] = result
    return l

movies_df['genres'] = movies_df['genres'].apply(remove_space)
movies_df['keywords'] = movies_df['keywords'].apply(remove_space)
movies_df['cast'] = movies_df['cast'].apply(remove_space)
movies_df.dropna(inplace=True)
movies_df['crew'] = movies_df['crew'].apply(remove_space)

movies_df['overview'] = movies_df['overview'].apply(lambda x: x.split())

movies_df['tags'] = movies_df['overview'] + movies_df['genres'] + movies_df['keywords'] + movies_df['cast'] + movies_df['crew']
movies_df['tags'] = movies_df['tags'].apply(lambda x: " ".join(x))
movies_df['tags'] = movies_df['tags'].apply(lambda x: x.lower())



from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(movies_df['tags']).toarray()

cv.get_feature_names_out()[:50]


similarity = cosine_similarity(vectors)
similarity.shape
def recommend(movie):
    rnum = movies_df.index[movies_df['title'].str.lower() == movie.lower()][0]
    data = list(enumerate(similarity[rnum]))
    sorted_data = sorted(data, key=lambda x: x[1], reverse=True)[1:6]
    for i in sorted_data:
        print(f"{movies_df.iloc[i[0]]['title']} : {i[1]}")


recommend('Avatar')
new_df = movies_df[['id', 'title', 'tags']]
similarity = similarity.astype("float32")

import pickle
pickle.dump(new_df, open('movies.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))


similarity.dtype

Titan A.E. : 0.2537477434955704
Small Soldiers : 0.25112360116696136
Independence Day : 0.24384310418681
Aliens vs Predator: Requiem : 0.24127175179050325
Ender's Game : 0.24012009007506566


/Users/anushragu/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/anushragu/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/anushragu/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


dtype('float32')